In [1]:
import json
from pathlib import Path

import numpy as np


def load_json(path):
    with open(path) as f:
        return json.load(f)


# ─── GPT-5 (from molecular_glue_extractor, experimental eval mode) ───
GPT5_ROOT = Path("/Users/yaochenr/project/molecular_glue_extractor/output/results")
GPT5_EVAL_DIRS = [GPT5_ROOT / d for d in ["260504_2123", "260504_2306", "260505_0037"]]

# ─── Gemini ───
GEMINI_ROOT = Path("/Users/yaochenr/project/tpd_curator/output/results")
GEMINI_EVAL_DIRS = [GEMINI_ROOT / d for d in ["260727_1429", "260826_1026", "260826_1430"]]

# ─── DeepSeek V3.1 ───
DS_ROOT = Path("/Users/yaochenr/project/tpd_curator/output/results")
DS_EVAL_DIRS = [DS_ROOT / d for d in ["260727_2159", "260812_1434", "260814_1519"]]

gpt5_raw = [load_json(d / "record_based_evaluation_results.json") for d in GPT5_EVAL_DIRS]
gemini_raw = [load_json(d / "record_based_evaluation_results.json") for d in GEMINI_EVAL_DIRS]
ds_raw = [load_json(d / "record_based_evaluation_results.json") for d in DS_EVAL_DIRS]

print(f"GPT-5     : {len(gpt5_raw)} replicates loaded")
print(f"Gemini    : {len(gemini_raw)} replicates loaded")
print(f"DeepSeek  : {len(ds_raw)} replicates loaded")

GPT-5     : 3 replicates loaded
Gemini    : 3 replicates loaded
DeepSeek  : 3 replicates loaded


In [2]:
PART_KEYS = [
    ("part1_record_level", "precision_records", "P (Part1)"),
    ("part1_record_level", "recall_records",    "R (Part1)"),
    ("part1_record_level", "f1_score_records",  "F1 (Part1)"),
    ("part2_entry_level",  "precision_entry",   "P (Part2)"),
    ("part2_entry_level",  "recall_entry",      "R (Part2)"),
    ("part2_entry_level",  "f1_score_entry",    "F1 (Part2)"),
]


def compute_mean_std(raw_list):
    """Compute mean ± std for each metric across replicates."""
    results = {}
    for section, key, name in PART_KEYS:
        vals = [r[section][key] for r in raw_list]
        results[name] = (np.mean(vals), np.std(vals, ddof=1))
    return results


gpt5_stats = compute_mean_std(gpt5_raw)
gemini_stats = compute_mean_std(gemini_raw)
ds_stats = compute_mean_std(ds_raw)

# ─── Print comparison table ───
print("═" * 80)
print(f"{'BACKBONE COMPARISON — Molecular Glue (n = 3 replicates each)':^80}")
print("═" * 80)
print()
print(f"{'Metric':<12} {'GPT-5':>18} {'Gemini':>18} {'DeepSeek V3.1':>18}")
print("─" * 80)

for _, _, name in PART_KEYS:
    g5_m, g5_s = gpt5_stats[name]
    gm_m, gm_s = gemini_stats[name]
    ds_m, ds_s = ds_stats[name]
    print(
        f"{name:<12} "
        f"{g5_m:.4f}±{g5_s:.4f}  "
        f"{gm_m:.4f}±{gm_s:.4f}  "
        f"{ds_m:.4f}±{ds_s:.4f}"
    )

print("─" * 80)
print()
print("Values: mean ± s.d. across 3 independent replicates")

════════════════════════════════════════════════════════════════════════════════
          BACKBONE COMPARISON — Molecular Glue (n = 3 replicates each)          
════════════════════════════════════════════════════════════════════════════════

Metric                    GPT-5             Gemini      DeepSeek V3.1
────────────────────────────────────────────────────────────────────────────────
P (Part1)    0.9777±0.0054  0.9228±0.0215  0.9866±0.0020
R (Part1)    0.9841±0.0275  0.9524±0.0412  0.9873±0.0110
F1 (Part1)   0.9785±0.0151  0.9259±0.0152  0.9863±0.0063
P (Part2)    0.9893±0.0186  0.9629±0.0239  0.9692±0.0134
R (Part2)    0.9769±0.0109  0.9612±0.0311  0.9172±0.0387
F1 (Part2)   0.9818±0.0056  0.9565±0.0075  0.9360±0.0294
────────────────────────────────────────────────────────────────────────────────

Values: mean ± s.d. across 3 independent replicates


In [3]:
# ─── Per-field entry-level breakdown ───

EXP_FIELDS = ["DC50", "DC50_units", "DC50_h", "Dmax", "Dmax_h", "Dmax_conc"]


def safe_div(a, b):
    return a / b if b > 0 else float("nan")


def prf(tp, fp, fn):
    p = safe_div(tp, tp + fp)
    r = safe_div(tp, tp + fn)
    f1 = safe_div(2 * p * r, p + r) if (p == p and r == r and (p + r) > 0) else float("nan")
    return p, r, f1


def per_field_stats(raw_list, fields=EXP_FIELDS):
    """Per-field micro-averaged metrics with replicate s.d."""
    out = {}
    for field in fields:
        ps, rs, fs = [], [], []
        for r in raw_list:
            pfm = r["part2_entry_level"]["per_field_metrics"].get(field)
            if pfm is None:
                continue
            tp, fp, fn = pfm["tp"], pfm["fp"], pfm["fn"]
            p_i, r_i, f1_i = prf(tp, fp, fn)
            ps.append(p_i)
            rs.append(r_i)
            fs.append(f1_i)
        out[field] = {
            "p_mean": np.mean(ps), "p_std": np.std(ps, ddof=1),
            "r_mean": np.mean(rs), "r_std": np.std(rs, ddof=1),
            "f1_mean": np.mean(fs), "f1_std": np.std(fs, ddof=1),
        }
    return out


gpt5_fields = per_field_stats(gpt5_raw)
gemini_fields = per_field_stats(gemini_raw)
ds_fields = per_field_stats(ds_raw)

print("═" * 100)
print(f"{'PER-FIELD F1 COMPARISON (Experimental Mode)':^100}")
print("═" * 100)
print(f"{'Field':<12} {'GPT-5':>18} {'Gemini':>18} {'DeepSeek V3.1':>18}")
print("─" * 100)

for field in EXP_FIELDS:
    g5 = gpt5_fields[field]
    gm = gemini_fields[field]
    ds = ds_fields[field]
    print(
        f"{field:<12} "
        f"{g5['f1_mean']:.4f}±{g5['f1_std']:.4f}  "
        f"{gm['f1_mean']:.4f}±{gm['f1_std']:.4f}  "
        f"{ds['f1_mean']:.4f}±{ds['f1_std']:.4f}"
    )

print("─" * 100)
print()

# Full P/R/F1 per field per backbone
for backbone_name, field_stats in [("GPT-5", gpt5_fields), ("Gemini", gemini_fields), ("DeepSeek V3.1", ds_fields)]:
    print(f"\n── {backbone_name} Per-field Details ──")
    print(f"{'Field':<12} {'Precision':>16} {'Recall':>16} {'F1':>16}")
    print("─" * 65)
    for field in EXP_FIELDS:
        s = field_stats[field]
        print(
            f"{field:<12} "
            f"{s['p_mean']:.4f}±{s['p_std']:.4f} "
            f"{s['r_mean']:.4f}±{s['r_std']:.4f} "
            f"{s['f1_mean']:.4f}±{s['f1_std']:.4f}"
        )
    print()

════════════════════════════════════════════════════════════════════════════════════════════════════
                            PER-FIELD F1 COMPARISON (Experimental Mode)                             
════════════════════════════════════════════════════════════════════════════════════════════════════
Field                     GPT-5             Gemini      DeepSeek V3.1
────────────────────────────────────────────────────────────────────────────────────────────────────
DC50         0.9981±0.0033  1.0000±0.0000  1.0000±0.0000
DC50_units   1.0000±0.0000  1.0000±0.0000  1.0000±0.0000
DC50_h       0.9562±0.0223  0.8994±0.0560  0.7955±0.2540
Dmax         1.0000±0.0000  0.9980±0.0034  0.9331±0.0580
Dmax_h       0.9920±0.0139  0.9550±0.0585  0.8261±0.1888
Dmax_conc    1.0000±0.0000  1.0000±0.0000  1.0000±0.0000
────────────────────────────────────────────────────────────────────────────────────────────────────


── GPT-5 Per-field Details ──
Field               Precision           Recall     